# C04 · DINOv3 forward pass on a T4 (check 8) + first timing (B5, preliminary)

Reproduces the 3 Nov check: **DINOv3 ViT-B/16 (LVD-1689M)** loads with your Hugging Face licence and a
224 px input gives an output of shape **[1, 201, 768]** (1 class token + 4 register tokens + 196 patch tokens).
Then it times 1024 px inputs in fp16, the tile size BRIGHT uses (no BRIGHT data needed: random tensors).

**Before running:** Runtime → Change runtime type → **T4 GPU**, and add `HF_TOKEN` (your Hugging Face token,
read access) in Colab Secrets with notebook access switched on. The model is gated, so the token must belong
to the account that accepted the DINOv3 licence.

In [ ]:
NOTEBOOK_NAME = "c04_dinov3_forward_pass"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
import torch, transformers, platform
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
from transformers import AutoModel
MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
model = AutoModel.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN")).eval().cuda()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"loaded {MODEL_ID}: {n_params:.1f} M parameters")

with torch.no_grad():
    out = model(pixel_values=torch.randn(1, 3, 224, 224, device="cuda"))
shape = list(out.last_hidden_state.shape)
print("output shape:", shape)
CHECK8_PASS = shape == [1, 201, 768]
print("CHECK 8:", "PASS" if CHECK8_PASS else "FAIL (expected [1, 201, 768])")

In [ ]:
# Preliminary B5 timing: 1024 px tiles (64 x 64 = 4096 patch tokens), fp16, batch sizes 1 and 2
import time, pandas as pd
rows = []
model_h = model.half()
for bs in (1, 2):
    x = torch.randn(bs, 3, 1024, 1024, device="cuda", dtype=torch.float16)
    try:
        with torch.no_grad():
            for _ in range(2):                      # warm-up
                model_h(pixel_values=x)
            torch.cuda.synchronize(); t0 = time.time(); n = 10
            for _ in range(n):
                model_h(pixel_values=x)
            torch.cuda.synchronize(); dt = time.time() - t0
        rows.append({"batch": bs, "tiles_per_s": round(bs * n / dt, 2),
                     "peak_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2)})
    except torch.cuda.OutOfMemoryError:
        rows.append({"batch": bs, "tiles_per_s": None, "peak_mem_GB": "OOM"})
    torch.cuda.empty_cache()
timing = pd.DataFrame(rows)
print(timing.to_string(index=False))
best = timing["tiles_per_s"].dropna().max()
if best == best:
    hours = 4246 / best / 3600
    print(f"All 4,246 BRIGHT tiles, one encoder: ~{hours:.2f} T4-hours; 3 encoders: ~{3*hours:.2f} T4-hours (NFR3 budget: 25)")

In [ ]:
import json
summary = {"model": MODEL_ID, "output_shape": shape, "check8_pass": bool(CHECK8_PASS),
           "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "transformers": transformers.__version__, "timing_1024px_fp16": rows}
with open(f"{RUN}/check8_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
timing.to_csv(f"{RUN}/dinov3_timing_1024px.csv", index=False)
print("saved to", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
print("damagesar-bench at commit", COMMIT)

In [ ]:
# Same model through the package's encoder interface (bench extract uses this path).
# Expect (14, 14, 768) for 224 px and (64, 64, 768) for a 1024 px tile.
import os, numpy as np
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from damagesar_bench.encoders import get_encoder
enc = get_encoder("dinov3_vitb16")
rng = np.random.default_rng(0)
g224 = enc.encode(rng.random((224, 224), dtype=np.float32))
g1024 = enc.encode(rng.random((1024, 1024), dtype=np.float32))
print("interface:", g224.shape, g1024.shape, g1024.dtype)
INTERFACE_PASS = g224.shape == (14, 14, 768) and g1024.shape == (64, 64, 768)
print("encoder interface:", "PASS" if INTERFACE_PASS else "FAIL")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/checks/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))